# 01 · Ingesta — Sofascore

Notebook de scraping de estadísticas de jugadores desde **Sofascore** usando la librería `ScraperFC`.

**Alcance:**
- 6 ligas: La Liga, Premier League, Serie A, Bundesliga, Ligue 1, Süper Lig
- Temporadas completas: 20/21, 21/22, 22/23, 23/24, 24/25
- Snapshot temporada actual: 25/26 (foto fija con fecha de descarga)

**Salida:** `data/raw/sofascore/df_<liga>_<temporada>.csv`

---

In [1]:
import pandas as pd
import ScraperFC as sfc

In [2]:
sf=sfc.Sofascore()

## 1. Exploración de ligas y temporadas disponibles

Se comprueba qué ligas y temporadas tiene disponibles la API de Sofascore
a través de ScraperFC, para confirmar la cobertura antes de lanzar el scraping.

In [3]:
print("--- LISTA DE LIGAS DISPONIBLES SOFASCORE ---")
for league_name in sfc.sofascore.comps.keys():
    print(league_name)

--- LISTA DE LIGAS DISPONIBLES SOFASCORE ---
Argentina Liga Profesional
Argentina Copa de la Liga Profesional
Bulgaria Parva Liga
CONCACAF Gold Cup
CONMEBOL Copa Libertadores
England Premier League
FIFA World Cup
FIFA Womens World Cup
France Ligue 1
Germany Bundesliga
Italy Serie A
Mexico Liga MX Apertura
Mexico Liga MX Clausura
Netherlands Eredivisie
Peru Liga 1
Portugal Primeira Liga
Saudi Arabia Pro League
Spain La Liga
Turkiye Super Lig
UEFA Champions League
UEFA Europa League
UEFA Conference League
UEFA European Championship
Ukraine Premier League
USA MLS
USA USL championship
USA USL League 1
USA USL Leauge 2


In [4]:
ligas_sofascore = [
    "Spain La Liga",
    "England Premier League",
    "Italy Serie A",
    "Germany Bundesliga",
    "France Ligue 1",
    "Turkiye Super Lig",
]

print("--- COMPROBANDO TEMPORADAS DISPONIBLES EN SOFASCORE ---")
for liga in ligas_sofascore:
    try:
        temporadas_sofascore = sf.get_valid_seasons(league=liga)
        lista_temporadas = list(temporadas_sofascore.keys())
        
        # Mostramos solo las 6 más recientes
        print(f"\n✅ {liga}:")
        print(f"   Últimas temporadas disponibles: {lista_temporadas[:6]}")
        
    except Exception as e:
        print(f"\n❌ Error al comprobar {liga}: {e}")

--- COMPROBANDO TEMPORADAS DISPONIBLES EN SOFASCORE ---
Running

✅ Spain La Liga:
   Últimas temporadas disponibles: ['25/26', '24/25', '23/24', '22/23', '21/22', '20/21']

✅ England Premier League:
   Últimas temporadas disponibles: ['25/26', '24/25', '23/24', '22/23', '21/22', '20/21']

✅ Italy Serie A:
   Últimas temporadas disponibles: ['25/26', '24/25', '23/24', '22/23', '21/22', '20/21']

✅ Germany Bundesliga:
   Últimas temporadas disponibles: ['25/26', '24/25', '23/24', '22/23', '21/22', '20/21']

✅ France Ligue 1:
   Últimas temporadas disponibles: ['25/26', '24/25', '23/24', '22/23', '21/22', '20/21']

✅ Turkiye Super Lig:
   Últimas temporadas disponibles: ['25/26', '24/25', '23/24', '22/23', '21/22', '20/21']


## 2. Scraping de temporadas completas (20/21 → 24/25)

Se descargan las estadísticas acumuladas de cada jugador por liga y temporada.
El parámetro `accumulation='total'` devuelve los totales de toda la temporada.
Se añaden las columnas `data_country` y `data_season` como metadatos de trazabilidad.

**SCRAPING TEMPORADAS 20/21,21/22,22/23,23/24,24/25**

In [5]:
# 1. Mapeo de las 6 ligas "temporada partida"
mapa_europa = {
    "Spain La Liga": "spain",
    "England Premier League": "england",
    "Italy Serie A": "italy",
    "Germany Bundesliga": "germany",
    "France Ligue 1": "france",
    "Turkiye Super Lig": "turkey"
}

# 2. Diccionario exacto: { Petición a Sofascore : Nombre final del archivo }
temporadas_europa = {
    '20/21': '2021',
    '21/22': '2122',
    '22/23': '2223',
    '23/24': '2324',
    '24/25': '2425'
}

print("Iniciando descarga de Ligas Europeas...")

for liga_completa, pais_corto in mapa_europa.items():
    for req_year, sufijo_archivo in temporadas_europa.items():
        
        file_name = f"df_{pais_corto}_{sufijo_archivo}.csv"
        
        try:
            print(f"📥 Descargando {liga_completa} ({req_year}) -> {file_name}...")
            
            df = sf.scrape_player_league_stats(year=req_year, league=liga_completa, accumulation='total')
            
            df['data_country'] = pais_corto
            df['data_season'] = sufijo_archivo
            
            df.to_csv(file_name, index=False)
            print(f"✅ Guardado: {file_name}")
            
        except Exception as e:
            print(f"❌ Error en {liga_completa} {req_year}: {e}")

Iniciando descarga de Ligas Europeas...
📥 Descargando Spain La Liga (20/21) -> df_spain_2021.csv...
✅ Guardado: df_spain_2021.csv
📥 Descargando Spain La Liga (21/22) -> df_spain_2122.csv...
✅ Guardado: df_spain_2122.csv
📥 Descargando Spain La Liga (22/23) -> df_spain_2223.csv...
✅ Guardado: df_spain_2223.csv
📥 Descargando Spain La Liga (23/24) -> df_spain_2324.csv...
✅ Guardado: df_spain_2324.csv
📥 Descargando Spain La Liga (24/25) -> df_spain_2425.csv...
✅ Guardado: df_spain_2425.csv
📥 Descargando England Premier League (20/21) -> df_england_2021.csv...
✅ Guardado: df_england_2021.csv
📥 Descargando England Premier League (21/22) -> df_england_2122.csv...
✅ Guardado: df_england_2122.csv
📥 Descargando England Premier League (22/23) -> df_england_2223.csv...
✅ Guardado: df_england_2223.csv
📥 Descargando England Premier League (23/24) -> df_england_2324.csv...
✅ Guardado: df_england_2324.csv
📥 Descargando England Premier League (24/25) -> df_england_2425.csv...
✅ Guardado: df_england_2425

## 3. Snapshot de la temporada actual (25/26)

La temporada 25/26 está aún en curso, por lo que se descarga como una foto fija
con la fecha del día en el nombre del archivo y en la columna `download_date`.
Este snapshot deberá actualizarse periódicamente hasta que finalicen las 6 ligas.

**SCRAPING SNAPSHOT TEMPORADA ACTUAL 25/26**

In [5]:
from datetime import datetime
fecha_hoy = datetime.today().strftime('%Y%m%d')
print(f"Iniciando Snapshot de la temporada actual ({fecha_hoy})...")

mapa_europa = {
    "Spain La Liga": "spain",
    "England Premier League": "england",
    "Italy Serie A": "italy",
    "Germany Bundesliga": "germany",
    "France Ligue 1": "france",
    "Turkiye Super Lig": "turkey"
}
req_year_eu = '25/26'
sufijo_eu = '2526'

for liga_completa, pais_corto in mapa_europa.items():
    file_name = f"df_{pais_corto}_{sufijo_eu}_snapshot_{fecha_hoy}.csv"
    try:
        print(f"📸 Snapshot {liga_completa} ({req_year_eu}) -> {file_name}")
        df = sf.scrape_player_league_stats(year=req_year_eu, league=liga_completa, accumulation='total')
        
        df['data_country'] = pais_corto
        df['data_season'] = sufijo_eu
        df['download_date'] = fecha_hoy # Marca de agua temporal
        
        df.to_csv(file_name, index=False)

    except Exception as e:
        print(f"❌ Error en {liga_completa}: {e}")

Iniciando Snapshot de la temporada actual (20260428)...
📸 Snapshot Spain La Liga (25/26) -> df_spain_2526_snapshot_20260428.csv
📸 Snapshot England Premier League (25/26) -> df_england_2526_snapshot_20260428.csv
📸 Snapshot Italy Serie A (25/26) -> df_italy_2526_snapshot_20260428.csv
📸 Snapshot Germany Bundesliga (25/26) -> df_germany_2526_snapshot_20260428.csv
📸 Snapshot France Ligue 1 (25/26) -> df_france_2526_snapshot_20260428.csv
📸 Snapshot Turkiye Super Lig (25/26) -> df_turkey_2526_snapshot_20260428.csv
